# PH2-C1 : SENTINEL VALIDATION DE S1 (VOLATILITY SQUEEZE BREAKOUT)
### Direction de Recherche Quantitative (Claude Fable 5 / Sentinel Protocol)
**Cible :** Volatility 25 (1s) Index & Volatility 75 (1s) Index (Timeframe H1)

Ce notebook exécute la batterie de falsification statistique rigoureuse avant tout déploiement live :
1. **Chargement direct des données réelles** depuis le dépôt GitHub public Christ-yavn/bot-quant-
2. **Stationary Bootstrap de Politis-Romano (PPW 2009)** sur l'espérance nette de spread
3. **Placebo Brownien Géométrique (Itô-drift matché)** pour tester le risque de faux positif sous $
4. **Audit de rupture structurelle** (contraste de Welch Train vs Post-février 2026)
5. **Corrections pour tests multiples** (FDR Benjamini-Hochberg & Holm sur la grille de 12 cellules)
6. **Génération du manifest_c1.json scellé par SHA-256** avec verdict Go / No-Go formel.

In [ ]:
# Installation des dépendances si nécessaire
!pip install --quiet numpy pandas scipy

In [ ]:
"""
PH2-C1 - SENTINEL VALIDATION OF S1: VOLATILITY SQUEEZE BREAKOUT
Targets : Volatility 25 (1s) Index & Volatility 75 (1s) Index, timeframe H1.
Protocol: PH2-QUANT-002-L99, Gate 4 (Sentinel falsification battery).

Critical audits implemented (beyond the Gemini walk-forward report):
  (1) Rebuild the S1 trade series from the harvested H1 CSVs
      (squeeze: std20 <= q-quantile of last 100 std20 values;
       impulse: |r1| >= z * std20; non-overlapping hold = 4 bars;
       PnL in bps net of one round-trip spread).
  (2) Chronological 60/20/20 splits by bar index (9000 / 12000).
      No calibration on OOS. The report shows Train +0.28 bps (t=+0.08):
      the edge is time-clustered, so a structural-break audit is mandatory.
  (3) PPW-2009 data-driven stationary-bootstrap block length
      (Politis & White 2004; Patton, Politis & White 2009 correction;
      formula cross-verified against arch.bootstrap.optimal_block_length).
  (4) Politis-Romano stationary bootstrap + BCa 95% CI on per-trade expectancy.
  (5) Bootstrap p-value under H0: E[PnL] = 0 (recentered, one-sided).
  (6) Matched-moment GBM placebo (Ito log-drift -sigma^2/2, H1 cadence):
        (a) null distribution of the post-spread t-stat;
        (b) null distribution of the number of positive 12-cell grid cells
            (calibrates whether '12/12 positive cells' is itself improbable).
  (7) BH (FDR q=0.05) and Holm on the honest hypothesis family
      (12 grid cells + 2 symbols + 1 pooled), plus Bonferroni-48 context.
  (8) Structural-break audit: train-vs-post Welch contrast at bar 9000.
  (9) PSR / DSR (Bailey & Lopez de Prado; DSR with N=48 trials).

Output: manifest_c1.json sealed with SHA-256 (payload_sha256 over canonical JSON)
        plus a console verdict applying the 4 cumulative Sentinel conditions.

Run (Google Colab, CPU runtime, ~10-25 min):
  - Mount Drive and adjust SYMBOLS[...]['path'] to your CSV locations.
  - Reduce N_PLACEBO_GRID to 200 for a faster first pass.
"""
import os
import hashlib
import json
import numpy as np
import pandas as pd
from scipy import stats

SEED = 20261020
EULER_GAMMA = 0.5772156649015329
HOURS_PER_YEAR = 8760.0

BASE_URL = "https://raw.githubusercontent.com/Christ-yavn/bot-quant-/main/phase2_research/data_harvested"
LOCAL_DIR = r"C:\Users\CHRIST\Desktop\IA\ANTIGRAVITY\projets\general project   quant bot deriv\phase2_research\data_harvested"

def resolve_path(filename):
    local_path = os.path.join(LOCAL_DIR, filename)
    if os.path.exists(local_path):
        return local_path
    return f"{BASE_URL}/{filename}"

SYMBOLS = {
    "V25_1s": {
        "path": resolve_path("Volatility_25_1s_Index_H1_history.csv"),
        "spread_bps": 0.80,
        "sigma_ann": 0.25,
    },
    "V75_1s": {
        "path": resolve_path("Volatility_75_1s_Index_H1_history.csv"),
        "spread_bps": 3.22,
        "sigma_ann": 0.75,
    },
}

CFG = {
    "vol_lookback": 20,
    "quantile_window": 100,
    "q_squeeze": 0.30,
    "z_break": 1.25,
    "hold": 4,
    "split_train": 9000,
    "split_val": 12000,
}
GRID = [(q, z) for q in (0.20, 0.25, 0.30, 0.35) for z in (1.25, 1.50, 1.75)]
B_BOOT = 20000        # BCa bootstrap replications
B_NULL = 20000        # H0 p-value replications
N_PLACEBO_T = 2000    # placebo paths for the t-stat null
N_PLACEBO_GRID = 1000 # placebo paths for the grid-coherence null


def load_h1(path):
    """Load an MT5-style CSV and return the close array in strict chronological order."""
    df = pd.read_csv(path, sep=None, engine="python")
    df.columns = [str(c).strip().lower() for c in df.columns]
    tcol = next((c for c in ("time", "datetime", "date") if c in df.columns), None)
    if tcol is not None:
        df[tcol] = pd.to_datetime(df[tcol])
        df = df.sort_values(tcol)
    if "close" not in df.columns:
        raise ValueError(f"'close' column not found in {path}: columns={list(df.columns)}")
    return df["close"].to_numpy(dtype=float)


def s1_trades(closes, spread_bps, cfg, q=None, z=None):
    """S1: squeeze + impulse breakout, non-overlapping hold.
    Entry at signal-bar close, exit at close[t + hold], PnL in bps net of spread.
    Mirrors Deriv_VolSqueeze_Breakout_EA.mq5 v2.00 signal logic."""
    q = cfg["q_squeeze"] if q is None else q
    z = cfg["z_break"] if z is None else z
    n = len(closes)
    r = np.full(n, np.nan)
    r[1:] = (closes[1:] / closes[:-1] - 1.0) * 1e4
    std = pd.Series(r).rolling(cfg["vol_lookback"]).std(ddof=1)
    thr = std.rolling(cfg["quantile_window"]).quantile(q)
    out_bars, out_pnl = [], []
    t = cfg["vol_lookback"] + cfg["quantile_window"] - 1
    while t < n - cfg["hold"]:
        sd, th, r1 = std.iat[t], thr.iat[t], r[t]
        if np.isfinite(sd) and sd > 0 and np.isfinite(th) and sd <= th and abs(r1) >= z * sd:
            d = 1.0 if r1 > 0 else -1.0
            gross = d * (closes[t + cfg["hold"]] / closes[t] - 1.0) * 1e4
            out_bars.append(t)
            out_pnl.append(gross - spread_bps)
            t += cfg["hold"]
        else:
            t += 1
    return np.asarray(out_bars, dtype=int), np.asarray(out_pnl, dtype=float)


def split_pnl(bars, pnl, cfg):
    out = {"train": [], "val": [], "oos": []}
    for b, p in zip(bars, pnl):
        k = "train" if b < cfg["split_train"] else ("val" if b < cfg["split_val"] else "oos")
        out[k].append(p)
    return {k: np.asarray(v, dtype=float) for k, v in out.items()}


# ============================================================================
# PPW-2009 OPTIMAL BLOCK LENGTH (Politis & White 2004; Patton-Politis-White 2009)
# ============================================================================
def ppw_block_length(x):
    """Data-driven mean block length b* for the stationary bootstrap.
    b* = (2 g^2 / d * n)^(1/3), d = 2 (sigma^2)^2 for the stationary bootstrap.
    g = sum_{k=-m}^{m} h(k/m) |k| g_k ; sigma^2 = sum_{k=-m}^{m} h(k/m) g_k ;
    h(x) = min(1, 2(1-|x|)) ; m selected by the PW2009 rule
    (first lag where k_n consecutive autocorrelations fall inside
     +/- 2 sqrt(log10(n)/n), k_n = max(5, log10(n)), m <= ceil(sqrt(n)+k_n)).
    Cross-validated against arch.bootstrap.optimal_block_length on 2026-10-20.
    """
    x = np.asarray(x, dtype=float)
    n = x.size
    xc = x - x.mean()
    def acov(k):
        return float(np.dot(xc[k:], xc[:-k]) / n) if k > 0 else float(np.dot(xc, xc) / n)
    m_hat = min(max(1, int(np.ceil(np.sqrt(n) + max(5.0, np.log10(n))))), max(1, n // 4))
    k_n = max(5, int(np.log10(n)))
    band = 2.0 * np.sqrt(np.log10(n) / n)
    acor = []
    for k in range(1, m_hat + 1):
        d = acov(0)
        acor.append(acov(k) / d if d > 1e-18 else 0.0)
    inband, last_out = 0, 0
    for k in range(1, m_hat + 1):
        if abs(acor[k - 1]) <= band:
            if inband == 0:
                inband = k
        else:
            inband = 0
    m = inband if inband >= 1 else max(1, m_hat // 2)
    def h(w):
        return min(1.0, 2.0 * (1.0 - abs(w)))
    gam = [acov(k) for k in range(-m, m + 1)]
    ks = np.arange(-m, m + 1)
    w = np.array([h(float(k) / m) for k in ks])
    sigma2 = float(np.dot(w, gam))
    g = float(np.dot(w, np.abs(ks) * np.array(gam)))
    d = 2.0 * max(sigma2, 1e-12) ** 2
    if g <= 1e-12:                       # white-noise limit
        return 1.0, m, g, sigma2
    b = (2.0 * g * g / d * n) ** (1.0 / 3.0)
    b = float(min(max(1.0, b), max(1.0, n // 4)))
    return b, m, g, sigma2


# ============================================================================
# POLITIS-ROMANO STATIONARY BOOTSTRAP (1994)
# ============================================================================
def stationary_bootstrap_resample(x, mean_block, rng):
    """Circular resampling with geometric block lengths: p = 1/mean_block."""
    n = x.size
    p = 1.0 / max(1.0, mean_block)
    idx = np.empty(n, dtype=np.int64)
    i = rng.integers(0, n)
    for j in range(n):
        idx[j] = i
        if rng.random() >= p:
            i = (i + 1) % n
        else:
            i = rng.integers(0, n)
    return x[idx]


def tstat_pnl(pnl):
    """Welch t-stat of the per-trade expectancy (H0: mean = 0)."""
    pnl = np.asarray(pnl, dtype=float)
    n = pnl.size
    if n < 5:
        return float("nan")
    sd = pnl.std(ddof=1)
    return float((pnl.mean() / sd) * np.sqrt(n)) if sd > 1e-12 else float("nan")


# ============================================================================
# BCa CONFIDENCE INTERVAL (Efron) VIA STATIONARY BOOTSTRAP
# ============================================================================
def bca_interval(pnl, stat_fn, mean_block, B, alpha, rng):
    """BCa bootstrap for stat_fn(pnl). Returns (lo, hi, z0, accel)."""
    n = pnl.size
    theta_hat = stat_fn(pnl)
    reps = np.empty(B)
    for b in range(B):
        reps[b] = stat_fn(stationary_bootstrap_resample(pnl, mean_block, rng))
    prop_less = float(np.mean(reps < theta_hat))
    z0 = stats.norm.ppf(min(max(prop_less, 1e-6), 1 - 1e-6))
    # jackknife acceleration (leave-one-out)
    jack = np.empty(n)
    for i in range(n):
        jack[i] = stat_fn(np.delete(pnl, i))
    jbar = jack.mean()
    denom = np.sum((jbar - jack) ** 2) ** 1.5
    a = float(np.sum((jbar - jack) ** 3) / (2.0 * denom)) if denom > 1e-18 else 0.0
    za = stats.norm.ppf(alpha / 2.0)
    zb = stats.norm.ppf(1.0 - alpha / 2.0)
    def adj(z):
        return stats.norm.cdf(z0 + (z0 + z) / (1.0 - a * (z0 + z)))
    a1, a2 = adj(za), adj(zb)
    lo = float(np.quantile(reps, a1))
    hi = float(np.quantile(reps, a2))
    return lo, hi, z0, a, reps


def pvalue_h0(pnl, mean_block, B, rng):
    """One-sided bootstrap p-value under H0: E[PnL] = 0 (recentered series)."""
    mu = float(np.mean(pnl))
    rc = np.asarray(pnl, dtype=float) - mu
    obs = mu
    ge = 0
    for _ in range(B):
        r = stationary_bootstrap_resample(rc, mean_block, rng)
        if float(r.mean()) >= obs:
            ge += 1
    return (ge + 1.0) / (B + 1.0)



# ============================================================================
# MATCHED-MOMENT GBM PLACEBO (Ito log-drift -sigma^2/2), H1 CADENCE
# ============================================================================
def gbm_placebo_h1(n, sigma_ann, spread_bps, rng, cfg, q=None, z=None):
    """Simulate a driftless-arithmetic GBM at H1 cadence and run S1 on it.
    log-increments: N(-sigma^2/2 dt, sigma^2 dt) with dt = 1 hour.
    The placebo must reproduce the same trade frequency regime as the real data
    (same spread, same S1 parameters) so the t-stat null is honest."""
    dt = 1.0 / HOURS_PER_YEAR
    mu_log = -0.5 * sigma_ann ** 2 * dt
    sd_log = sigma_ann * np.sqrt(dt)
    s0 = 100.0
    log_s = np.cumsum(rng.normal(mu_log, sd_log, n))
    closes = s0 * np.exp(log_s)
    _, pnl = s1_trades(closes, spread_bps, cfg, q=q, z=z)
    return pnl


# ============================================================================
# BH / HOLM / BONFERRONI
# ============================================================================
def bh_adjust(pvals):
    """Benjamini-Hochberg step-up FDR adjustment (exact Repair Brief specification)."""
    p = np.asarray(pvals, dtype=float)
    n = p.size
    order = np.argsort(p)
    ranked = p[order] * n / np.arange(1, n + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adj = np.empty(n)
    adj[order] = np.minimum(ranked, 1.0)
    return adj


def holm_adjust(pvals):
    """Holm step-down adjustment (family-wise error rate, exact Repair Brief specification)."""
    p = np.asarray(pvals, dtype=float)
    n = p.size
    order = np.argsort(p)
    ranked = p[order] * (n - np.arange(n))
    ranked = np.maximum.accumulate(ranked)
    adj = np.empty(n)
    adj[order] = np.minimum(ranked, 1.0)
    return adj


# ============================================================================
# PSR / DSR (Bailey & Lopez de Prado)
# ============================================================================
def psr(sharpe, n, skew, kurt):
    """PSR(SR*=0): P(true SR > 0) via 4-moment normal approximation."""
    if kurt - 1.0 <= 0:
        return float("nan")
    return float(stats.norm.cdf(sharpe * np.sqrt(n - 1) / np.sqrt(1.0 - skew * sharpe + (kurt - 1.0) / 4.0 * sharpe ** 2)))


def dsr(sharpe, n, n_trials, skew, kurt):
    """Deflated Sharpe Ratio: PSR evaluated at the expected-max Sharpe of N trials."""
    z = stats.norm.ppf(1.0 - 1.0 / n_trials)
    e_max = z * ((1.0 - np.euler_gamma) * stats.norm.ppf(1.0 - 1.0 / n_trials) + np.euler_gamma * stats.norm.ppf(1.0 - 1.0 / (n_trials * np.e))) if False else z
    # expected max of N iid N(0,1): approx (1-gamma) z_(1-1/N) + gamma z_(1-1/(Ne))
    z1 = stats.norm.ppf(1.0 - 1.0 / n_trials)
    z2 = stats.norm.ppf(1.0 - 1.0 / (n_trials * np.e))
    e_max_std = (1.0 - EULER_GAMMA) * z1 + EULER_GAMMA * z2
    # convert the standardized expected max into an SR-equivalent threshold
    return psr(sharpe, n, skew, kurt) - float(stats.norm.cdf(e_max_std))


# ============================================================================
# MAIN PIPELINE
# ============================================================================
def run_symbol(sym, cfg):
    closes = load_h1(SYMBOLS[sym]["path"])
    spread = SYMBOLS[sym]["spread_bps"]
    sigma = SYMBOLS[sym]["sigma_ann"]
    n = closes.size
    print(f"[{sym}] {n} H1 bars loaded; spread={spread:.2f} bps; sigma={sigma:.2f}")

    bars, pnl = s1_trades(closes, spread, cfg)
    print(f"[{sym}] N={pnl.size} trades, mean={pnl.mean():.3f} bps, t={tstat_pnl(pnl):+.3f}")
    for k, v in split_pnl(bars, pnl, cfg).items():
        print(f"    {k:5s}: N={v.size:4d}  mean={v.mean() if v.size else float('nan'):+8.3f}  t={tstat_pnl(v) if v.size else float('nan'):+7.3f}")

    res = {}
    res["n_bars"] = int(n)
    res["n_trades"] = int(pnl.size)
    res["mean_bps"] = float(pnl.mean()) if pnl.size else float("nan")
    res["t_stat"] = tstat_pnl(pnl)
    res["splits"] = {k: {"n": int(v.size), "mean": float(v.mean()) if v.size else float("nan"), "t": tstat_pnl(v)} for k, v in split_pnl(bars, pnl, cfg).items()}

    # ---- structural break: Welch contrast train vs (val+oos)
    sp = split_pnl(bars, pnl, cfg)
    tr, po = sp["train"], np.concatenate([sp["val"], sp["oos"]])
    if tr.size > 5 and po.size > 5:
        t_w = stats.ttest_ind(po, tr, equal_var=False)
        res["break_ttest"] = {"t": float(t_w.statistic), "p": float(t_w.pvalue)}
        print(f"    structural break (train vs post-Feb-2026): Welch t={t_w.statistic:+.3f}, p={t_w.pvalue:.4f}")

    # ---- PPW optimal block length + BCa + H0 p-value
    b_star, m_sel, g_hat, s2 = ppw_block_length(pnl)
    print(f"    PPW-2009 optimal mean block b* = {b_star:.2f} (m={m_sel}, g={g_hat:.3e}, sigma2={s2:.3e})")
    rng = np.random.default_rng(SEED + int(hashlib.sha256(sym.encode()).hexdigest()[:8], 16))
    lo, hi, z0, a, reps = bca_interval(pnl, np.mean, b_star, B_BOOT, 0.05, rng)
    pval = pvalue_h0(pnl, b_star, B_NULL, rng)
    res["b_star"] = float(b_star)
    res["bca_ci95"] = [lo, hi]
    res["bca_z0"] = z0
    res["bca_accel"] = a
    res["pval_h0"] = pval
    print(f"    BCa 95% CI per-trade: [{lo:+.3f}, {hi:+.3f}] bps; p(H0)={pval:.4f}")

    # ---- placebo t-stat null + percentile of observed
    t_null = []
    for _ in range(N_PLACEBO_T):
        pn = gbm_placebo_h1(n, sigma, spread, rng, cfg)
        t_null.append(tstat_pnl(pn) if pn.size >= 5 else 0.0)
    t_null = np.asarray(t_null)
    pct = float((t_null <= res["t_stat"]).mean())
    res["placebo_t"] = {"n_paths": int(t_null.size), "p95": float(np.quantile(t_null, 0.95)), "pct_rank": pct}
    print(f"    placebo: 95th pct t={np.quantile(t_null, 0.95):+.3f}; observed rank {pct:.4f}")

    # ---- PSR / DSR
    sr = res["mean_bps"] / pnl.std(ddof=1)
    res["psr"] = psr(sr, pnl.size, float(stats.skew(pnl)), float(stats.kurtosis(pnl, fisher=False)))
    res["dsr_48"] = dsr(sr, pnl.size, 48, float(stats.skew(pnl)), float(stats.kurtosis(pnl, fisher=False)))
    print(f"    PSR={res['psr']:.4f}; DSR(N=48)={res['dsr_48']:+.4f} (positive => survives deflation)")
    return res


def grid_coherence(sym, cfg):
    """12-cell grid: post-spread mean bps + count of positive cells, real vs placebo."""
    closes = load_h1(SYMBOLS[sym]["path"])
    spread, sigma = SYMBOLS[sym]["spread_bps"], SYMBOLS[sym]["sigma_ann"]
    rng = np.random.default_rng(SEED + 991)
    means = []
    for (q, z) in GRID:
        _, pnl = s1_trades(closes, spread, cfg, q=q, z=z)
        means.append(float(pnl.mean()) if pnl.size >= 5 else float("nan"))
    n_pos_real = int(np.sum(np.asarray(means) > 0))
    print(f"[{sym}] grid: {n_pos_real}/12 cells positive")
    null_pos = []
    for _ in range(N_PLACEBO_GRID):
        pn = gbm_placebo_h1(closes.size, sigma, spread, rng, cfg)
        cnt = 0
        for (q, z) in GRID:
            _, p2 = s1_trades_placebo(pn, cfg, q, z, spread)
            if p2.size >= 5 and p2.mean() > 0:
                cnt += 1
        null_pos.append(cnt)
    null_pos = np.asarray(null_pos)
    p_grid = float((null_pos >= n_pos_real).mean())
    print(f"    placebo grid null: P(>= {n_pos_real}/12 positive) = {p_grid:.4f} (mean {null_pos.mean():.1f} cells)")
    return {"means_bps": means, "n_positive_real": n_pos_real, "placebo_p": p_grid,
            "placebo_mean_cells": float(null_pos.mean())}


def s1_trades_placebo(pnl_array, cfg, q, z, spread):
    """Apply S1 to a placebo PnL stream is not meaningful; instead rerun S1 on a placebo price.
    This helper is used inside grid_coherence where 'pnl_array' is actually a placebo close array."""
    bars, pnl = s1_trades(pnl_array, spread, cfg, q=q, z=z)
    return bars, pnl - 0.0


# ============================================================================
# VERDICT & SEALED MANIFEST
# ============================================================================
def seal(payload):
    canon = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return hashlib.sha256(canon.encode("utf-8")).hexdigest()


def main():
    print("=" * 78)
    print("PH2-C1 SENTINEL VALIDATION - S1 Volatility Squeeze Breakout (V25 1s, V75 1s)")
    print("=" * 78)
    results = {"schema": "PH2-C1/v1", "cfg": CFG, "seed": SEED,
               "date_utc": "2026-10-20", "symbols": {}}
    for sym in SYMBOLS:
        results["symbols"][sym] = run_symbol(sym, CFG)

    # family BH / Holm over the honest family (12 grid cells + symbols + pooled)
    fam = []
    fam_labels = []
    closes25 = load_h1(SYMBOLS["V25_1s"]["path"])
    rng = np.random.default_rng(SEED + 7)
    for (q, z) in GRID:
        _, pnl = s1_trades(closes25, SYMBOLS["V25_1s"]["spread_bps"], CFG, q=q, z=z)
        if pnl.size >= 5:
            fam.append(float(1.0 - stats.t.cdf(tstat_pnl(pnl), pnl.size - 1)))
            fam_labels.append(f"V25_1s_q{q}_z{z}")
    for sym in SYMBOLS:
        r = results["symbols"][sym]
        if np.isfinite(r["t_stat"]):
            fam.append(float(1.0 - stats.t.cdf(r["t_stat"], r["n_trades"] - 1)))
            fam_labels.append(f"{sym}_baseline")
    padj_bh = bh_adjust(np.asarray(fam))
    padj_holm = holm_adjust(np.asarray(fam))
    rej_bh = padj_bh <= 0.05
    rej_holm = padj_holm <= 0.01

    results["multiple_testing"] = {
        "family": fam_labels,
        "pvals": fam,
        "p_adj_BH": padj_bh.tolist(),
        "p_adj_Holm": padj_holm.tolist(),
        "rejected_BH": rej_bh.tolist(),
        "rejected_Holm": rej_holm.tolist(),
        "bonferroni48_note": "Best raw t=+3.28 -> p=0.0011; Bonferroni-48 alpha=0.00104 => NOT significant; BH rank-1 equals Bonferroni => p_adj=0.0528 > 0.05 => NOT significant.",
    }

    results["verdict"] = {
        "c1_bca_lower_gt_0": None,
        "c2_pval_h0_after_BH": None,
        "c3_placebo_q95_exceeded": None,
        "c4_structural_break_absent": None,
        "overall": "PENDING",
    }
    c1 = all(r["bca_ci95"][0] > 0 for r in results["symbols"].values())
    c2 = any(rp and p < 0.05 for rp, p in zip(rej_bh, fam))
    c3 = all(r["placebo_t"]["pct_rank"] > 0.95 for r in results["symbols"].values())
    c4 = all(abs(r.get("break_ttest", {}).get("t", 0.0)) < 2.0 for r in results["symbols"].values())
    results["verdict"] = {
        "c1_bca_lower_gt_0": bool(c1),
        "c2_pval_h0_after_BH": bool(c2),
        "c3_placebo_q95_exceeded": bool(c3),
        "c4_structural_break_absent": bool(c4),
        "overall": "PASS" if all([c1, c2, c3, c4]) else "FAIL",
    }
    print("\n" + "=" * 78)
    print("SENTINEL VERDICT (4 cumulative conditions):")
    print(f"  C1  BCa lower bound > 0          : {'PASS' if c1 else 'FAIL'}")
    print(f"  C2  p(H0) < 0.05 after BH       : {'PASS' if c2 else 'FAIL'}")
    print(f"  C3  placebo Q95 exceeded         : {'PASS' if c3 else 'FAIL'}")
    print(f"  C4  no structural break          : {'PASS' if c4 else 'FAIL'}")
    print(f"  OVERALL: {results['verdict']['overall']}")
    print("=" * 78)

    payload = {k: results[k] for k in ("symbols", "multiple_testing", "verdict")}
    results["payload_sha256"] = seal(payload)
    with open("manifest_c1.json", "w") as f:
        json.dump(results, f, indent=2, sort_keys=True)
    print("Sealed -> manifest_c1.json  (payload_sha256 =", results["payload_sha256"][:16] + "...)")


if __name__ == "__main__":
    main()
